In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import seaborn as sns
from scipy.stats import mannwhitneyu

# Get the current working directory
current_directory = os.getcwd()
# If required go to repository root
if os.path.split(current_directory)[1] != 'PAM_Parametrization':
    # Go up two levels
    parent_directory = os.path.dirname(os.path.dirname(current_directory))
    # Change the directory to the parent directory
    os.chdir(parent_directory)
    
from PAModelpy.utils import set_up_pam
from Scripts.pam_generation import setup_ecoli_pam as set_up_ecoli_pam_curated
from Modules.utils.sector_config_functions import change_translational_sector_with_config_dict
from Modules.utils.pamparametrizer_analysis import get_results_from_simulations
from Modules.utils.pam_generation import create_pamodel_from_diagnostics_file

# from Modules.utils import calculate_r_squared_for_reaction
# from Scripts.Visualization.PAMparametrizer_progress_cleaned_figure import run_simulations

N_ALT_MODELS = 8
    
ECOLI_PHENOTYPE_DATA_PATH = os.path.join('Data', 'Ecoli_phenotypes')
MODEL_FILE_PATH = os.path.join('Models', 'iML1515.xml')

PARAM_FILE_OLD = os.path.join('Results', '1_preprocessing','proteinAllocationModel_iML1515_EnzymaticData_241209.xlsx')
PARAM_FILE_SCALED = os.path.join('Results', '2_parametrization','proteinAllocationModel_iML1515_EnzymaticData_241209_multi.xlsx')

PARAMETER_RESULT_FILES = [os.path.join('Results','3_analysis','parameter_files',
                                     f'proteinAllocationModel_EnzymaticData_iML1515_{i}_1.xlsx') for i in range(0,N_ALT_MODELS+1)]
BEST_INDIV_RESULT_FILES = [os.path.join('Results','2_parametrization','diagnostics',
                                     f'pam_parametrizer_diagnostics_{i}_1.xlsx') for i in range(0,N_ALT_MODELS+1)]

# 1. Load reference data

In [4]:
# load exchange rates for different carbon sources by Gerosa et al. (2015) in Ecoli BW25113
flux_csources = pd.read_excel(os.path.join(ECOLI_PHENOTYPE_DATA_PATH, 'Ecoli_phenotypes_py.xls'),
                       sheet_name = 'Fluxes_Csources',
                            engine='openpyxl',
                            index_col=1)
flux_csources_df = flux_csources.drop(['Flux (publication)', 'Reversibility'], axis=1)
flux_csources_df.head()

,Acetate,Fructose,Galactose,Glucose,Glycerol,Gluconate,Pyruvate,Succinate,Glucose (flux ratio Glc)
Reaction identifier,,,,,,,,,
EX_ac_e_b,13.584,-3.32866,-1.968939e-08,-6.827019,-0.597000,-5.003982,-11.91391,-3.320974,-0.70717
EX_fru_e_b,0.000,8.32800,0.000000e+00,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000
EX_gal_e_b,0.000,0.00000,1.969000e+00,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000
EX_glc__D_e_b,0.000,0.00000,0.000000e+00,9.654000,0.000000,0.000000,0.00000,0.000000,1.00000
EX_glyc_e_b,0.000,0.00000,0.000000e+00,0.000000,4.944834,0.000000,0.00000,0.000000,0.00000


# 2. Setup the *Escherichia coli* iML1515 model with new parameters

In [5]:
#setup the model
ecoli_pam_wt = set_up_pam(PARAM_FILE_OLD, 
                          model = MODEL_FILE_PATH, 
                          sensitivity=False) # not curation for reference
ecoli_pam_curated = set_up_ecoli_pam_curated(
    pam_data_file_path = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
    sensitivity = False) # curated for reference

pam = set_up_pam(PARAM_FILE_SCALED, 
                 model = MODEL_FILE_PATH,
                 sensitivity = False)
pam.change_reaction_bounds('EX_glc__D_e', lower_bound=0)

new_ecoli_pams = {alt+1: create_pamodel_from_diagnostics_file(file,
                                          pam.copy(copy_with_pickle=True)) for alt, file in enumerate(BEST_INDIV_RESULT_FILES)}

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector

Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedEnzymeSector

Done with setting up the proteome allocation model iML1515

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector



/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/PAModel.py:247: UserWarning: Molar mass for E332 is invalid: 0.0
  warnings.warn(f"Molar mass for {enz.id} is invalid: {molmass}")


Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedProteinSector

Done with setting up the proteome allocation model iML1515

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector

Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedEnzymeSector

Done with setting up the proteome allocation model iML1515



/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SS is not associated with enzyme O32583_P0A6B7_P30138_P30140_P77718. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ARBTptspp is not associated with enzyme P08839_P0AA04_P17334_P69791_P69795. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction GAMptspp is not associated with enzyme P08839_P0AA04_P36672_P69783. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction AACPS3 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SS is not associated with enzyme O32583_P0A6B7_P30138_P30140_P77718. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ARBTptspp is not associated with enzyme P08839_P0AA04_P17334_P69791_P69795. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction GAMptspp is not associated with enzyme P08839_P0AA04_P36672_P69783. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction AACPS3 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is

...

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction S2FE2SS2 is not associated with enzyme P77499_P77522_P77667_P77689. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction AKGDH is not associated with enzyme P06959_P0A9P0_P0AFG8. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SS is not associated with enzyme O32583_P0A6B7_P30138_P30140_P77718. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ARBTptspp is not associated with enzyme P08839_P0AA04_P17334_P69791_P69795. Skip
  warn(f"Reaction {rxn_

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction S2FE2SS2 is not associated with enzyme P76194_P77444. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ACPPAT181 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction POR5 is not associated with enzyme P0ABY4_P28903. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction DSBDR is not associated with enzyme P36655_P77202. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralv

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction MNLptspp is not associated with enzyme P08839_P0AA04_P36672_P69783. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction G3PAT141 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction DSBDR is not associated with enzyme P0A9P4_P0AGG4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SR is not associated with enzyme P0AAC8_P0ACD4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip"

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ACMUMptspp is not associated with enzyme P08839_P0AA04_P36672_P69783. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction G3PAT120 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction RNDR1 is not associated with enzyme P0A9P4_P0AGG4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SS2 is not associated with enzyme P0AAC8_P0ACD4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Sk

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction ACMUMptspp is not associated with enzyme P08839_P0AA04_P36672_P69783. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction G3PAT120 is not associated with enzyme P0A6A8_P0A722. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction RNDR1 is not associated with enzyme P0A9P4_P0AGG4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Skip")
/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/Enzyme.py:257: UserWarning: Reaction I2FE2SS2 is not associated with enzyme P0AAC8_P0ACD4. Skip
  warn(f"Reaction {rxn_id} is not associated with enzyme {self.id}. Sk

In [6]:
pam.catalytic_events.query('')[0].enzymes

[<EnzymeComplex P0A6B7_P0ACD4 at 0x7feb5f2f5250>]

# 3. Check internal flux distribution

In [ ]:
def calculate_error_for_reactions(validation_df: pd.DataFrame,
                                   flux_df: pd.DataFrame,
                                  rxns_to_validate: list) -> float:
    # calculate error for different exchange rates
    error = []
    for rxn, substrate_id in zip(rxns_to_validate, flux_df.substrate_id):
        # only select the rows which are filled with data
        validation_data = validation_df.dropna(axis=0, subset=rxn)
        validation_data = validation_data.loc[substrate_id]
        # if there are no reference data points, continue to the next reaction
        if len(validation_data) == 0:
            continue
        r_squared = calculate_r_squared_for_reaction(rxn, validation_df, substrate_id,
                                                           flux_df[flux_df.substrate_id == substrate_id])
        error += [r_squared]
    return error

def calculate_r_squared_for_reaction(reaction_id: str, validation_data: pd.DataFrame,
                                     substrate_uptake_id: str,
                                      fluxes: pd.DataFrame) -> float:
    substr_rxn = substrate_uptake_id
    # Take the absolute value of substrate uptake to avoid issues with reaction directionality
    validation_data[substr_rxn] = [round(abs(flux),4) for flux in validation_data[substr_rxn]]
    simulated_data = pd.DataFrame({substr_rxn: [round(abs(flux),4) for flux in fluxes['substrate']],
                                   'simulation': fluxes[reaction_id]})
    ref_data_rxn = pd.merge(validation_data,simulated_data,on=substr_rxn, how='inner')
    # error: squared difference
    ref_data_rxn = ref_data_rxn.assign(error=lambda x: (x[reaction_id] - x['simulation']) ** 2)

    # calculate R^2:
    data_average = np.nanmean(validation_data[reaction_id])
    residual_ss = np.nansum(ref_data_rxn.error)
    total_ss = np.nansum([(data - data_average) ** 2 for data in ref_data_rxn[reaction_id]])
    # calculating r_squared is only feasible of the numerator and the denomenator are both nonzero
    if (residual_ss == 0) | (total_ss == 0):
        r_squared = 0
    else:
        r_squared = 1 - residual_ss / total_ss
    return r_squared

def calculate_difference_simulation_experiment(validation_df, flux_df, rxns_to_validate, substr_rxn):
    differences = []
    for rxn in rxns_to_validate:
        # only select the rows which are filled with data
        validation_data = validation_df.dropna(axis=0, subset=rxn)
        # if there are no reference data points, continue to the next reaction
        if len(validation_data) == 0:
            continue
        validation_data[substr_rxn] = [round(abs(flux),4) for flux in validation_data[substr_rxn]]
        simulated_data = pd.DataFrame({substr_rxn: [round(abs(flux_df['substrate']),4)],
                                   'simulation': flux_df[rxn]})
        ref_data_rxn = pd.merge(validation_data,simulated_data,on=substr_rxn, how='inner')
        # error: squared difference
        differences += [row[rxn] - row['simulation'] if not np.isnan(row.simulation) else row[rxn] for i,row in ref_data_rxn.iterrows()]
    return differences

## 3.1 Parse dataframes to validate the flux distribution

In [ ]:
# Get the data for growth on multiple carbon sources from Gerosa et al. (2015)
# Make sure all the fluxes of backward reactions are inverted to match the model directionality
fluxes_to_simulate = flux_csources_df.copy()
new_indices = []
for i, row in fluxes_to_simulate.iterrows():
    if isinstance(row.name, str):
        if row.name[-2:] == '_b':
            new_indices.append(row.name[:-2])
            fluxes_to_simulate.loc[row.name]= -row
        else: 
            new_indices.append(row.name)
    else:
        new_indices.append(row.name)
            
fluxes_to_simulate.index = new_indices
fluxes_to_simulate_parsed = fluxes_to_simulate[fluxes_to_simulate.index.notnull()]
fluxes_to_simulate_parsed = fluxes_to_simulate_parsed.rename(
    index = {'BIOMASS_Ec_iML1515_WT_75p37M':'BIOMASS_Ecoli_core_w_GAM'}
).drop('Glucose (flux ratio Glc)', axis = 1)

In [ ]:
# extract the validation data and substrate information for each carbon source
flux_mapper = {col: fluxes_to_simulate_parsed.index[i] for i,col in enumerate(fluxes_to_simulate_parsed.columns)}
fluxes_to_save = []
# Get the fluxes we want to save
for flux in fluxes_to_simulate_parsed.index:
    if isinstance(flux, str):
        fluxes_to_save += [f for f in flux.split(', ')]

#parse the fluxes such that we can run and validate simulations easily
validation_df = pd.DataFrame(columns = list(fluxes_to_simulate_parsed.index))
substrate_ids = []
substrate_uptake = []
for substrate, fluxes in fluxes_to_simulate_parsed.items():
    substrate_ids += [flux_mapper[substrate]]
    substrate_uptake += [fluxes.loc[flux_mapper[substrate]]]
    validation_df = pd.concat([validation_df,fluxes.to_frame().T], ignore_index =True)

validation_df.index = list(flux_mapper.values())
validation_df

## 3.2 Run simulations

In [ ]:
kwargs = {'substrate_ids': list(validation_df.index), 
          'substrate_rates': [[rate] for rate in substrate_uptake],
          'fluxes_to_save' : fluxes_to_save}
# for each study, run simulations
fluxes_curated = get_results_from_simulations(ecoli_pam_curated, **kwargs)['fluxes']
print('\n')
fluxes_wt = get_results_from_simulations(ecoli_pam_wt,**kwargs)['fluxes']
print('\n')
# fluxes_new = run_simulations(ecoli_pam_new, **kwargs)

fluxes_new_dict = {alt: get_results_from_simulations(pam, **kwargs)['fluxes'] for alt, pam in new_ecoli_pams.items()}

In [ ]:
#calculate R^2 value between simulations and experiments
error_curated = calculate_error_for_reactions(validation_df,
                                                 fluxes_curated,
                                                 fluxes_to_save)
print('R^2 values for the model with published parameter set :', np.nanmean(error_curated))

error_wt = calculate_error_for_reactions(validation_df,
                                                 fluxes_wt,
                                                 fluxes_to_save[1:])
print('R^2 values for the model with parameters from GotEnzymes: ', np.nanmean(error_wt))


# error_new = calculate_error_for_reactions(validation_df,
#                                                  fluxes_new,
#                                                  fluxes_to_save[1:])
error_new_dict = {alt: calculate_error_for_reactions(validation_df,
                                                  fluxes,
                                                  fluxes_to_save[1:]) for alt, fluxes in fluxes_new_dict.items()}
for alt, error_list in error_new_dict.items():
    print(f'R^2 values for alternative model {alt} with the optimized parameters: ', np.nanmean(error_list))



## 3.2 Visualize the simulation results for the different models

In [ ]:
# validation_df_1.index = validation_df.index.str.split(', ')
validation_df_1 = validation_df.T.reset_index()
validation_df_1['index'] = validation_df_1['index'].str.split(', ')
validation_df_1 = validation_df_1.explode('index').set_index('index').T
validation_df_1
# validation_df = validation_df.explode()

### experiment vs simulation scatterplot

In [ ]:
models = ['GotEnzymes','Curated'] + [f'PAMparametrizer {alt}' for alt in range(1,N_ALT_MODELS+1)]
model_colors = sns.color_palette("coolwarm", n_colors=len(models))
cmap = dict(zip(models, model_colors))
fontsize=15

fig, axs = plt.subplots(nrows = int(len(fluxes_new_dict)/2), ncols = 2, sharey='row', sharex='col', figsize = (20,20))
axs = axs.flatten()

for alt, fluxes_new in fluxes_new_dict.items():
    ax = axs[alt-1]
    curated = []
    wt = []
    new = []
    validation = []
    for i in range(len(fluxes_curated)):
        curated += [abs(flx) for flx in fluxes_curated.iloc[i].to_list()[1:-1]]
        if i<len(fluxes_wt):
            wt += [abs(flx) for flx in fluxes_wt.iloc[i].to_list()[1:-1]]
        else:
            wt += [0]*len(fluxes_curated.iloc[i].to_list()[1:-1])
        if i<len(fluxes_new):
            new += [abs(flx) for flx in fluxes_new.iloc[i].to_list()[1:-1]]
        else:
            new += [0]*len(fluxes_curated.iloc[i].to_list()[1:-1])
        validation += validation_df_1.iloc[i].abs().to_list()
    
    ax.plot([0,25], [0,25], linestyle = 'dashed', color='grey')
    ax.scatter(curated, validation, label = 'curated', color = 'black')
    ax.scatter(wt, validation, label = 'GotEnzymes', color ='blue')
    ax.scatter(new, validation, color = cmap[f'PAMparametrizer {alt}'])#, label = f'PAMparametrizer {alt}'
    ax.set_title(f'PAMparametrizer {alt}', fontsize = fontsize)

fig.supxlabel('simulated flux [$mmol/g_{CDW}/h$]', fontsize = fontsize*1.5)
fig.supylabel('measured flux [$mmol/g_{CDW}/h$]', fontsize = fontsize*1.5)
plt.legend()

plt.tight_layout()
plt.show()

### Boxplots of simulation error

In [ ]:
def plot_significant_annotation(ax, column, sign, compare_df, dodge_factor):
        # Add annotation for significance
        x1, x2 = 1, column
        y, h = compare_df.Difference.quantile(0.88)+ dodge_factor, 0.75  # Dynamically adjust y position
        ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, color='k')
        ax.text((x1 + x2) * 0.5, y + h*0.6, sign, ha='center', va='bottom', fontsize=fontsize, color='k')

In [ ]:
fontsize = 15
models = ['GotEnzymes', 'Curated'] + [f'alternative {alt}' for alt in error_new_dict.keys()]
model_colors = sns.color_palette("coolwarm", n_colors=len(models))
cmap = dict(zip(models, model_colors))

# Prepare the figure
fig, ax = plt.subplots(figsize=(12, 6))

# Combine data into a DataFrame
all_differences = pd.DataFrame()
curated_differences = None  # Placeholder for Curated errors
num_significant = 0

for col, (model, sub_df) in enumerate(zip(models, [fluxes_wt, fluxes_curated] + list(fluxes_new_dict.values()))):
    differences = []
    for _, row in sub_df.iterrows():
        substrate_id = row['substrate_id']
        difference = calculate_difference_simulation_experiment(
            validation_df_1, row, fluxes_to_save[1:], substrate_id)
        differences += difference

    temp_df = pd.DataFrame({'Model': [model] * len(differences), 'Difference': differences})
    
    # Store curated differences for comparison
    if model == 'GotEnzymes':
        curated_differences = differences
        curated_diff_df = temp_df[temp_df.Model == 'GotEnzymes']
    elif 'PAMparametrizer' in model or 'alternative' in model:
        # Statistical test
        stat, p = mannwhitneyu(curated_differences, differences, alternative='greater')
        print(f"{model}: U-statistic = {stat}, p-value = {p}")
#         if p < 0.075:
#             sign = '*'
#         if p < 0.05:
#             sign = '**'
#         if p < 0.01:
#             sign = '***'
#         if p < 0.075:
#             num_significant += 1
#             plot_significant_annotation(ax, col, sign, curated_diff_df, 1+num_significant*0.65)

    # Append to the main DataFrame
    all_differences = pd.concat([all_differences, temp_df], ignore_index=True)

# Boxplot or Violin Plot
sns.boxplot(x='Model', y='Difference', data=all_differences, ax=ax, palette=cmap, showfliers=False)

# Adjust y-axis to focus on bulk data and add space for annotations
ax.set_ylim([all_differences['Difference'].quantile(0.05)-3, all_differences['Difference'].quantile(0.95)+3])

# Set labels and title
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=fontsize)
ax.set_xlabel('Model', fontsize=fontsize * 1.5)
ax.set_ylabel(r'Difference (exp - sim) mmol/$\text{g}_{CDW}$/h', fontsize=fontsize)

plt.tight_layout()
# plt.show()
plt.savefig('Results/3_analysis/multiple_csources_error_boxplot.png')

### Histograms of simulation error

In [ ]:
models = ['GotEnzymes','Curated'] + [f'alternative {alt}' for alt in error_new_dict.keys()]
model_colors = sns.color_palette("coolwarm", n_colors=len(models))
cmap = dict(zip(models, model_colors))
n_bins=25

fig, ax = plt.subplots(ncols = 3,figsize=(15, 5), dpi=100)
row = -1

for model in models:
    data = all_differences[all_differences.Model == model]
    
    ax[0].hist(data.Difference, label = model, color = to_hex(cmap[model]), histtype = 'step')#histogram
    ax[1].hist(data.Difference, bins=n_bins, cumulative=True, density=True, histtype='step', 
            label=model, color=to_hex(cmap[model]), linewidth=2) #cumulative histogram
    sns.violinplot(x='Model', y='Difference', data=all_differences, palette=cmap, ax=ax[2])


# Set labels and title
ax[0].set_yscale('log')
ax[0].set_xlabel(r'Difference (exp - sim) mmol/$\text{g}_{CDW}$/h', fontsize=fontsize * 1.5)
ax[0].set_ylabel('Frequency', fontsize=fontsize * 1.5)

plt.legend()
plt.tight_layout()
plt.show()

### Kernel density plot
A Kernel Density Plot (KDE Plot) is a statistical tool used to estimate the probability density function of a continuous variable. It provides a smooth, continuous curve that represents the distribution of your data, offering a visualization of where data points are concentrated.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5), dpi=100)

for model in models:
    data = all_differences[all_differences.Model == model]['Difference']
    if model == 'Curated': color = 'black'
    elif model == 'GotEnzymes': color = 'grey'
    else: color = to_hex(cmap[model])
    sns.kdeplot(data, ax=ax, label=model, color=color, linewidth=2)

# Set labels and title
ax.set_xlabel(r'Difference (exp - sim) mmol/$\text{g}_{CDW}$/h', fontsize=fontsize * 1.5)
ax.set_ylabel('Density', fontsize=fontsize * 1.5)
ax.set_xlim([-20,20])
ax.legend()
plt.tight_layout()
plt.show()

### line graphs of simulation per reaction

In [ ]:
# visualize per flux
fig, axs = plt.subplots(ncols = 5, nrows = 6, figsize = [20,20])
substrate_ids_cur = fluxes_curated.substrate_id
substrate_ids = fluxes_new_dict[1].substrate_id


#make the plot panels for each reaction
fig_reactions = []
for i in range(0,36,6):
    fig_reactions += [[rxn for rxn in fluxes_to_save[i:i+5]]]
# plot all reactions
for j in range(6):
    reactions_to_plot = fig_reactions[j]
    for i, rxn in enumerate(reactions_to_plot):
        validation = validation_df_1[rxn]
        axs[j,i].scatter(substrate_ids_cur, validation_df_1.loc[substrate_ids_cur, rxn].abs(), color = 'grey')
        axs[j,i].plot(substrate_ids_cur, fluxes_curated[rxn].abs(), label = 'curated', color = 'black')
        axs[j,i].plot(fluxes_wt['substrate_id'], fluxes_wt[rxn].abs(),label = 'GotEnzymes', color = 'blue')
        for alt, fluxes in fluxes_new_dict.items():
            axs[j,i].plot(fluxes['substrate_id'], fluxes[rxn].abs(), label = f'alternative {alt}', 
                          color = cmap[f'alternative {alt}'])

#         axs[j,i].plot(substrate_ids, fluxes_new[rxn].abs(), label = 'PAMparametrizer', color = 'red')
        axs[j,i].set_title(rxn)
        axs[j,i].tick_params(axis='x', labelrotation=60)

        
# Shrink current axis's height by 10% on the bottom
box = axs[j,i].get_position()
axs[j,i].set_position([box.x0, box.y0 + box.height * 0.1,
                 box.width, box.height * 0.9])
        
handles, labels = axs[j,i].get_legend_handles_labels()    
fig.legend(handles, labels, loc = 'lower center', bbox_to_anchor=(0.5, -0.05),ncols = 8, fontsize= fontsize)

fig.supylabel('flux rate [mmol/gCDW/h]', fontsize = fontsize*1.5)
fig.supylabel('limiting nutrient', fontsize = fontsize*1.5)

plt.tight_layout()

plt.show()